# Exploración visual del dataset sintético

Notebook para validar que los datos generados cumplen las especificaciones.

In [11]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("synthetic_api_activity.csv", parse_dates=["timestamp_local"])
df["hour"] = df["timestamp_local"].dt.hour + df["timestamp_local"].dt.minute / 60
df["day_of_week"] = df["timestamp_local"].dt.dayofweek
df["is_weekend"] = df["day_of_week"] >= 5
df["date"] = df["timestamp_local"].dt.date
df["week"] = df["timestamp_local"].dt.isocalendar().week.astype(int)

print(f"Registros: {len(df)}")
print(f"Rango: {df['timestamp_local'].min()} → {df['timestamp_local'].max()}")
df.describe()

Registros: 25920
Rango: 2025-01-01 00:00:00 → 2025-03-31 23:55:00


,timestamp_local,actividad,http_4xx,http_5xx,sum_downloadTime,avg_downloadTime,hour,day_of_week,week
count,25920,25920.000000,25920.000000,25920.000000,2.592000e+04,25920.000000,25920.000000,25920.000000,25920.000000
mean,2025-02-14 23:57:30,406.786188,4.734375,0.466281,1.039696e+05,246.558483,11.958333,3.022222,7.211111
min,2025-01-01 00:00:00,1.000000,0.000000,0.000000,1.240000e+02,91.600000,0.000000,0.000000,1.000000
25%,2025-01-23 11:58:45,92.000000,1.000000,0.000000,2.130675e+04,217.035000,5.979167,1.000000,4.000000
50%,2025-02-14 23:57:30,323.000000,3.000000,0.000000,7.697000e+04,243.110000,11.958333,3.000000,7.000000
75%,2025-03-09 11:56:15,696.000000,7.000000,0.000000,1.678062e+05,268.982500,17.937500,5.000000,10.000000
max,2025-03-31 23:55:00,4479.000000,190.000000,628.000000,4.099914e+06,2716.040000,23.916667,6.000000,14.000000
std,NaN,346.925050,5.524087,10.769148,1.097948e+05,75.762277,6.928295,1.999915,3.722360


## 1. Serie temporal completa de actividad

Visión general de los 3 meses. Debería verse el patrón diurno repetido y la caída en fines de semana.

In [12]:
fig = px.line(df, x="timestamp_local", y="actividad", title="Actividad - Serie temporal completa")
fig.update_layout(xaxis_title="Fecha", yaxis_title="Actividad (peticiones/5min)", height=400)
fig.show()

## 2. Semana tipo (una semana concreta)

Zoom a una semana laboral + fin de semana para ver el ciclo diurno claramente.

In [13]:
week_df = df[(df["timestamp_local"] >= "2025-01-06") & (df["timestamp_local"] < "2025-01-13")]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["Actividad", "http_4xx"])
fig.add_trace(go.Scatter(x=week_df["timestamp_local"], y=week_df["actividad"],
                         mode="lines", name="actividad"), row=1, col=1)
fig.add_trace(go.Scatter(x=week_df["timestamp_local"], y=week_df["http_4xx"],
                         mode="lines", name="http_4xx", line=dict(color="orange")), row=2, col=1)
fig.update_layout(height=500, title_text="Semana 6-12 enero 2025")
fig.show()

## 3. Perfil diurno medio (laborable vs fin de semana)

Media de actividad por franja horaria. Debe mostrar el valle de madrugada y el pico a mediodía.

In [14]:
hourly = df.groupby(["is_weekend", df["timestamp_local"].dt.hour]).agg(
    actividad_mean=("actividad", "mean"),
    actividad_std=("actividad", "std")
).reset_index()
hourly.columns = ["is_weekend", "hour", "actividad_mean", "actividad_std"]
hourly["tipo"] = hourly["is_weekend"].map({False: "Laborable", True: "Fin de semana"})

fig = px.line(hourly, x="hour", y="actividad_mean", color="tipo",
              title="Perfil diurno medio de actividad")
fig.update_layout(xaxis_title="Hora del día", yaxis_title="Actividad media", height=400)
fig.show()

## 4. Ratio http_4xx / actividad

Debe estar en torno al 1%. Se muestra la distribución y la evolución temporal.

In [15]:
df["ratio_4xx"] = df["http_4xx"] / df["actividad"] * 100

fig = make_subplots(rows=1, cols=2, subplot_titles=["Distribución del ratio 4xx (%)", "Ratio 4xx diario (%)"])

fig.add_trace(go.Histogram(x=df["ratio_4xx"], nbinsx=50, name="ratio_4xx"), row=1, col=1)

daily_ratio = df.groupby("date").apply(lambda g: g["http_4xx"].sum() / g["actividad"].sum() * 100).reset_index()
daily_ratio.columns = ["date", "ratio_4xx"]
fig.add_trace(go.Scatter(x=daily_ratio["date"], y=daily_ratio["ratio_4xx"],
                         mode="lines+markers", name="ratio diario"), row=1, col=2)

fig.update_layout(height=400, title_text=f"http_4xx: ratio global = {df['http_4xx'].sum()/df['actividad'].sum()*100:.2f}%")
fig.show()

C:\Users\nicol\AppData\Local\Temp\ipykernel_11808\2262650344.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ratio = df.groupby("date").apply(lambda g: g["http_4xx"].sum() / g["actividad"].sum() * 100).reset_index()


## 5. http_5xx

Debe ser prácticamente 0 siempre, con picos esporádicos muy raros.

In [16]:
fig = px.scatter(df[df["http_5xx"] > 0], x="timestamp_local", y="http_5xx",
                 size="http_5xx", title=f"Ocurrencias de http_5xx ({(df['http_5xx']>0).sum()} de {len(df)} registros)")
fig.update_layout(height=350, xaxis_title="Fecha", yaxis_title="http_5xx")
fig.show()

## 6. Tiempos de descarga

`sum_downloadTime` debe seguir la tendencia de actividad. `avg_downloadTime` debe ser relativamente estable.

In [17]:
week_df = df[(df["timestamp_local"] >= "2025-02-03") & (df["timestamp_local"] < "2025-02-10")]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["actividad", "sum_downloadTime", "avg_downloadTime"])

fig.add_trace(go.Scatter(x=week_df["timestamp_local"], y=week_df["actividad"],
                         mode="lines", name="actividad"), row=1, col=1)
fig.add_trace(go.Scatter(x=week_df["timestamp_local"], y=week_df["sum_downloadTime"],
                         mode="lines", name="sum_downloadTime", line=dict(color="green")), row=2, col=1)
fig.add_trace(go.Scatter(x=week_df["timestamp_local"], y=week_df["avg_downloadTime"],
                         mode="lines", name="avg_downloadTime", line=dict(color="red")), row=3, col=1)

fig.update_layout(height=650, title_text="Semana 3-9 febrero: actividad vs tiempos de descarga")
fig.show()

## 7. Correlación actividad ↔ sum_downloadTime

In [18]:
fig = px.scatter(df.sample(3000, random_state=0), x="actividad", y="sum_downloadTime",
                 opacity=0.3, trendline="ols",
                 title="Correlación actividad vs sum_downloadTime (muestra 3k puntos)")
fig.update_layout(height=400)
fig.show()

print(f"Correlación Pearson: {df['actividad'].corr(df['sum_downloadTime']):.4f}")

Correlación Pearson: 0.8622


## 8. Heatmap: actividad media por hora y día de la semana

In [19]:
pivot = df.pivot_table(values="actividad", index=df["timestamp_local"].dt.hour,
                        columns="day_of_week", aggfunc="mean")
pivot.columns = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

fig = px.imshow(pivot, aspect="auto", color_continuous_scale="YlOrRd",
                title="Actividad media por hora y día de la semana",
                labels=dict(x="Día", y="Hora", color="Actividad media"))
fig.update_layout(height=500)
fig.show()